# 01 — Data Audit

**Goal:** establish exactly what data is available for the two parts of this project.

- **Part 1 (Messi era):** StatsBomb open event data — which La Liga seasons are covered, and which event types are relevant to the offside trap study.
- **Part 2 (Flick era):** what FBref and Understat expose for the 2024/25 and 2025/26 seasons.

No metrics are computed here. This notebook only answers: *what do we have, and what can we measure with it?*

In [ ]:
import pandas as pd
from statsbombpy import sb

pd.set_option('display.max_columns', None)

## Part A — StatsBomb open data (Messi era)

### A1. Which La Liga seasons are available?

In [ ]:
competitions = sb.competitions()
la_liga = competitions[competitions['competition_name'] == 'La Liga'][['season_id', 'season_name']]
la_liga = la_liga.drop_duplicates().sort_values('season_name').reset_index(drop=True)
print(f"{len(la_liga)} La Liga seasons available")
la_liga

### A2. Load all Barcelona La Liga matches

We pull every match where Barcelona appeared (home or away) across all available seasons.

In [ ]:
COMPETITION_ID = 11
BARCA = 'Barcelona'

all_matches = []
for _, row in la_liga.iterrows():
    m = sb.matches(competition_id=COMPETITION_ID, season_id=row['season_id'])
    m['season_name'] = row['season_name']
    all_matches.append(m)

matches = pd.concat(all_matches, ignore_index=True)
barca_matches = matches[
    (matches['home_team'] == BARCA) | (matches['away_team'] == BARCA)
].copy()

print(f"Total La Liga matches in open data : {len(matches)}")
print(f"Barcelona fixtures                  : {len(barca_matches)}")
print(f"Seasons spanned                     : {barca_matches['season_name'].nunique()}")

### A3. Inspect event types in a single match

Before writing any metric, we look at the raw event types StatsBomb records. We want to confirm:
- Is there an explicit **Offside** event type?
- What **pass types** exist that could represent balls played over/through the line?
- What **defensive action** event types exist (for measuring line height via action location)?

We load one match as a representative sample.

In [ ]:
sample = barca_matches.iloc[0]
events = sb.events(match_id=sample['match_id'])

print(f"Sample match: {sample['home_team']} vs {sample['away_team']} ({sample['season_name']})")
print(f"Total events in match: {len(events)}\n")
print("Event types present:")
print(events['type'].value_counts().to_string())

### A4. Inspect offside-related events

Two mechanisms can signal an offside trap success in StatsBomb data:
1. An explicit `Offside` event — the simplest case.
2. A `Pass` event with `pass_outcome == 'Pass Offside'` — a pass flagged for offside at the recipient.

We check both.

In [ ]:
offside_events = events[events['type'] == 'Offside']
print(f"Explicit 'Offside' events in this match: {len(offside_events)}")
if len(offside_events):
    display(offside_events[['minute', 'team', 'player']].head(10))

print()

pass_offside = events[
    (events['type'] == 'Pass') & (events['pass_outcome'] == 'Pass Offside')
]
print(f"Passes with outcome 'Pass Offside': {len(pass_offside)}")
if len(pass_offside):
    display(pass_offside[['minute', 'team', 'player', 'pass_length', 'pass_technique']].head(10))

### A5. Inspect defensive action events and their locations

To proxy defensive line height, we use the x-location of Barcelona's defensive actions (tackles, interceptions, clearances). A high average location = a high defensive line. We confirm these event types and their coordinate columns exist.

In [ ]:
defensive_types = ['Tackle', 'Interception', 'Clearance', 'Block']
barca_def = events[
    (events['type'].isin(defensive_types)) & (events['team'] == BARCA)
]

print(f"Barcelona defensive actions in this match: {len(barca_def)}")
print(f"\nCoordinate columns available: {[c for c in barca_def.columns if 'location' in c.lower()]}")
print("\nAction type counts:")
print(barca_def['type'].value_counts().to_string())

barca_def[['type', 'minute', 'location']].head(6)

## Part B — FBref / Understat (Flick era)

StatsBomb open data ends in 2020/21. For Flick's Barcelona (2024/25–2025/26) we rely on aggregated sources.

### B1. FBref — per-match defensive stats

FBref blocks all programmatic access via Cloudflare. We download the CSV manually and commit it to `data/external/` so the analysis remains reproducible.

**To get the file (one-time, ~2 min):**
1. Open in your browser: `https://fbref.com/en/squads/206d90db/2024-2025/matchlogs/c12/defense/Barcelona-Match-Logs-La-Liga`
2. Above the table click **"Share & more" → "Get table as CSV"**
3. Copy all text and save as `data/external/fbref_barca_defense_2024_25.csv`

In [ ]:
from pathlib import Path

CSV_PATH = Path('../data/external/fbref_barca_defense_2024_25.csv')

if not CSV_PATH.exists():
    print("File not found. Follow the instructions in the cell above to download it.")
else:
    defense_log = pd.read_csv(CSV_PATH, header=[0, 1])
    print(f"Rows: {len(defense_log)}")
    print(f"\nColumns available:")
    print(defense_log.columns.tolist())

In [ ]:
defense_log.head(5)

## Summary — what we can measure

| Dimension | Messi era (StatsBomb) | Flick era (FBref/Understat) |
|---|---|---|
| Offsides drawn per match | ✓ direct from `Pass Offside` events | ✓ aggregated in FBref match log |
| Pressing intensity (PPDA) | ✓ computed from events | Proxy via progressive passes allowed |
| Defensive action height | ✓ from event x/y coordinates | ✗ not available |
| Pass types that beat the line | ✓ pass technique + sequence | ✗ not available |
| Shot locations conceded | ✓ from shot events | ✓ Understat (xG + location) |

**Next notebook (`02_messi_era_offside.ipynb`):** extract offsides drawn and PPDA for every Barcelona La Liga match in the Messi era, and begin testing the press–line hypothesis.